# BTS — Etapa B3: XGBoost original en SageMaker (4 M + 1 M + 1 M)

Continúa **exactamente después de B2**. Lee los mismos 120 Parquet de la etapa A; no vuelve a descargar Silver, reconstruir variables, cambiar muestras ni reentrenar B1/B2.

**Fidelidad al notebook original:** utiliza `xgb.XGBClassifier`, `tree_method="hist"`, categóricas nativas, los mismos hiperparámetros y la misma parada temprana sobre `logloss` de validación. Se reutilizan los niveles categóricos del checkpoint B1, derivados exclusivamente de entrenamiento. El único cambio de representación para ahorrar RAM es convertir columnas numéricas a `float64`, igual que en B1. Esto puede causar diferencias numéricas menores respecto de ejecuciones locales anteriores.

**Memoria:** se cargan *train* y *val* con las columnas necesarias, se entrena y guarda el modelo, se generan probabilidades de validación, se libera *train/val* y solo entonces se carga *test*. Esto reduce el pico de memoria, **pero no garantiza** que XGBoost con 4 M de filas quepa en 16 GiB. Si el kernel se reinicia, no vuelvas a ejecutar B1/B2 ni reduzcas muestras automáticamente: revisaremos el pico de RAM para adaptar B3.

**Rutas:** ejecutar desde `/home/ec2-user/SageMaker/src/` en el kernel Python 3. No ejecutes la celda de respaldo S3 antes de completar las predicciones y métricas.

**Variante experimental `float64`:** usa exactamente los Parquet de la etapa A, mismas muestras, columnas, niveles categóricos, semillas y parámetros del modelo. Solo cambia la conversión numérica `np.float32 → np.float64`, el nombre del checkpoint y su prefijo S3; los resultados originales `float32` permanecen intactos. Un cambio en el orden local original de la muestra puede seguir afectando las cifras, por lo que esta prueba aísla *solo el efecto de la conversión numérica en AWS*.


In [4]:
%pip install -q xgboost pandas numpy pyarrow scikit-learn boto3

Note: you may need to restart the kernel to use updated packages.


## 0. Configuración y rutas

In [5]:
from pathlib import Path
import gc, json, shutil, sys, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import xgboost as xgb
import boto3

ROOT = Path.cwd()
assert ROOT.name == "src", f"Ejecuta desde SageMaker/src. Carpeta actual: {ROOT}"
PARTS_BASE = Path("../data/checkpoints/features_seed42_h240_chunk50000")
B1_DIR = Path("../outputs/outputs_modeling/checkpoints/lightgbm_float64")
OUT_DIR = Path("../outputs/outputs_modeling/checkpoints/xgboost_float64")
OUT_DIR.mkdir(parents=True, exist_ok=True)
BUCKET = "bts-flights-analysis"
B1_PREFIX = "modeling/checkpoints/lightgbm-float64-original-seed42-h240"
S3_PREFIX = "modeling/checkpoints/xgboost-float64-original-seed42-h240"
s3 = boto3.client("s3")

SEED, TARGET = 42, "DepDel15"
N_TRAIN, N_EVAL = 4_000_000, 1_000_000
COST_FP, COST_FN = 1.0, 3.0
LGB_MAX_TREES, LGB_EARLY_STOP = 3000, 100  # nombres originales: compartidos con XGBoost

print("Etapa A:", PARTS_BASE.resolve())
print("Salida B3:", OUT_DIR.resolve())
print("Disco libre:", round(shutil.disk_usage(ROOT).free / 1024**3, 2), "GiB")
print("XGBoost:", xgb.__version__)

Etapa A: /home/ec2-user/SageMaker/data/checkpoints/features_seed42_h240_chunk50000
Salida B3: /home/ec2-user/SageMaker/outputs/outputs_modeling/checkpoints/xgboost_float64
Disco libre: 25.39 GiB
XGBoost: 3.2.0


## 1. Verificar los Parquet de etapa A (solo metadatos)

In [6]:
PARTS = {}
for split, expected in {"train": N_TRAIN, "val": N_EVAL, "test": N_EVAL}.items():
    folder = PARTS_BASE / split
    paths = sorted(folder.glob("part_*.parquet"))
    assert (folder / "_SUCCESS").exists(), f"Falta _SUCCESS: {folder}"
    assert paths, f"Faltan Parquet en {folder}"
    total = sum(pq.ParquetFile(p).metadata.num_rows for p in paths)
    assert total == expected, f"{split}: {total:,} != {expected:,}"
    PARTS[split] = paths
    print(f"✔ {split}: {len(paths)} Parquet, {total:,} filas")

✔ train: 80 Parquet, 4,000,000 filas
✔ val: 20 Parquet, 1,000,000 filas
✔ test: 20 Parquet, 1,000,000 filas


## 2. Mismas variables y categorías que LightGBM original

El código de modelado define `FULL_FEATURES` y ajusta las categorías usando **solo el entrenamiento**. B1 ya guardó esos niveles en `feature_schema.json`. Se reutilizan, evitando recorrer otra vez el entrenamiento para identificar categorías.

In [7]:
schema_path = B1_DIR / "feature_schema.json"
if not schema_path.exists():
    B1_DIR.mkdir(parents=True, exist_ok=True)
    s3.download_file(BUCKET, f"{B1_PREFIX}/feature_schema.json", str(schema_path))
    print("Esquema B1 recuperado de S3")
SCHEMA = json.loads(schema_path.read_text())
CAT_COLS = ["Reporting_Airline", "Origin", "Dest", "route", "OriginState", "DestState", "DepTimeBlk"]
NUM_STATIC = ["Distance", "CRSElapsedTime", "dep_min", "arr_min", "Month", "DayOfWeek", "DayofMonth",
              "day_of_year", "is_weekend", "days_to_holiday"]
HIST_KEYS = ["origin", "route", "airline", "tail", "origin_hr"]
HIST_COLS = [f"{k}_rate_{s}" for k in HIST_KEYS for s in ("all", "30d")]
SCHED_COLS = ["origin_dep_load", "origin_day_load", "dest_arr_load", "tail_leg_num", "turnaround_min"]
STATUS_COLS = ["prev_leg_delayed", "tail_delayed_known", "tail_known_legs", "origin_known_today", "origin_delay_rate_today"]
FULL_FEATURES = NUM_STATIC + CAT_COLS + HIST_COLS + SCHED_COLS + STATUS_COLS
assert SCHEMA["feature_names"] == FULL_FEATURES, "El esquema de B1 difiere del modelado original"
assert SCHEMA["cat_cols"] == CAT_COLS, "Las categóricas difieren del modelado original"
assert SCHEMA["prediction_horizon_min"] == 240, "Horizonte diferente de 240 minutos"
CAT_LEVELS = SCHEMA["cat_levels"]
NUMERIC_COLS = [c for c in FULL_FEATURES if c not in CAT_COLS]
COLS_TO_READ = FULL_FEATURES + [TARGET]
missing = set(COLS_TO_READ) - set(pq.ParquetFile(PARTS["train"][0]).schema_arrow.names)
assert not missing, f"Faltan columnas en etapa A: {sorted(missing)}"
(OUT_DIR / "feature_schema.json").write_text(json.dumps(SCHEMA, indent=2, ensure_ascii=False))
print("✔ Esquema compatible:", len(FULL_FEATURES), "features;", len(CAT_COLS), "categorías")

✔ Esquema compatible: 37 features; 7 categorías


## 3. Carga compacta, con orden de filas preservado

No se cambia el muestreo ni el conjunto: se leen los Parquet de etapa A ordenados por nombre. Las categóricas conservan **exactamente los niveles de entrenamiento**, necesarios para XGBoost nativo.

In [8]:
def load_compact(split):
    t0 = time.time()
    frames = []
    for i, path in enumerate(PARTS[split], 1):
        frame = pd.read_parquet(path, columns=COLS_TO_READ)
        for col in NUMERIC_COLS:
            frame[col] = pd.to_numeric(frame[col], errors="coerce").astype(np.float64)
        for col in CAT_COLS:
            frame[col] = pd.Categorical(frame[col], categories=CAT_LEVELS[col])
        frame[TARGET] = frame[TARGET].astype(np.uint8)
        frames.append(frame)
        if i % 20 == 0:
            print(f"  {split}: {i}/{len(PARTS[split])} fragmentos")
    result = pd.concat(frames, ignore_index=True)
    del frames
    gc.collect()
    print(f"✔ {split}: {len(result):,} filas | pandas {result.memory_usage(deep=True).sum()/1024**3:.2f} GiB | {time.time()-t0:.0f}s")
    return result


def check_b1_labels(split, y):
    """Confirma alineación con las etiquetas guardadas en B1."""
    path = B1_DIR / f"{split}_predictions.npz"
    if not path.exists():
        B1_DIR.mkdir(parents=True, exist_ok=True)
        s3.download_file(BUCKET, f"{B1_PREFIX}/{split}_predictions.npz", str(path))
        print("Recuperado desde S3:", path.name)
    with np.load(path) as b1:
        expected = b1["y"]
        assert y.shape == expected.shape and np.array_equal(y, expected), (
            f"Etiquetas de {split} no coinciden con B1. No comparar métricas hasta corregir el orden."
        )
    print(f"✔ {split}: etiquetas coinciden con B1")

## 4. Entrenar XGBoost original (4 M) y guardar modelo + validación

Se conservan **todos los hiperparámetros** del bloque `7.3 XGBoost` del notebook original: 3000 árboles máximos, learning rate 0.05, profundidad 7, `hist`, categorías nativas y *early stopping* de 100 rondas con log loss. No modifica la regla de entrenamiento.

Si el kernel se reinicia **en `model.fit`**, ya no es un problema de construcción de variables: será necesario adaptar el almacenamiento interno de XGBoost manteniendo las filas, no repetir A/B1/B2.

In [9]:
train = load_compact("train")
val = load_compact("val")
y_train = train.pop(TARGET).to_numpy(dtype=np.uint8, copy=False)
y_val = val.pop(TARGET).to_numpy(dtype=np.uint8, copy=False)
check_b1_labels("val", y_val)
assert list(train.columns) == FULL_FEATURES and list(val.columns) == FULL_FEATURES
assert all(train[c].dtype == np.float64 for c in NUMERIC_COLS), "Train no es float64"
assert all(val[c].dtype == np.float64 for c in NUMERIC_COLS), "Val no es float64"

# Parámetros copiados del notebook de modelado original, sección 7.3.
XGB_PARAMS = dict(
    n_estimators=LGB_MAX_TREES, learning_rate=0.05, max_depth=7, min_child_weight=100,
    subsample=0.8, colsample_bytree=0.7, reg_lambda=20.0,
    tree_method="hist", enable_categorical=True, max_cat_to_onehot=4,
    eval_metric=["auc", "logloss"], early_stopping_rounds=LGB_EARLY_STOP,
    n_jobs=-1, random_state=SEED,
)

model = xgb.XGBClassifier(**XGB_PARAMS)
print("Entrenando XGBoost con", len(train), "filas")
t0 = time.time()
model.fit(train, y_train, eval_set=[(val, y_val)], verbose=False)
training_seconds = round(time.time() - t0, 2)
best_iteration = int(model.best_iteration) + 1
print(f"✔ XGBoost: {best_iteration} árboles | {training_seconds:.1f}s")

# UBJSON preserva las divisiones categóricas nativas; el JSON es compatible con el modo nativo.
model.save_model(str(OUT_DIR / "model.ubj"))
p_val = model.predict_proba(val, iteration_range=(0, best_iteration))[:, 1]
np.savez_compressed(OUT_DIR / "val_predictions.npz", y=y_val, p=p_val)

run_config = {
    "model": "XGBoost", "seed": SEED,
    "train_rows": len(train), "val_rows": len(val), "test_rows": N_EVAL,
    "train_end": "2024-12-31", "val_end": "2025-08-31", "prediction_horizon_min": 240,
    "training_seconds": training_seconds, "best_iteration": best_iteration,
    "params": XGB_PARAMS,
    "representation": "float64 numeric + pandas categorical with levels fitted in B1 float64 train",
    "versions": {"xgboost": xgb.__version__, "numpy": np.__version__, "pandas": pd.__version__},
}
(OUT_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2, ensure_ascii=False))
print("✔ Modelo, configuración y predicciones de validación guardados")

# Liberar train y val ANTES de leer el millón de prueba.
del train, val, y_train, y_val, p_val
gc.collect()

  train: 20/80 fragmentos
  train: 40/80 fragmentos
  train: 60/80 fragmentos
  train: 80/80 fragmentos
✔ train: 4,000,000 filas | pandas 0.94 GiB | 7s
  val: 20/20 fragmentos
✔ val: 1,000,000 filas | pandas 0.23 GiB | 2s
✔ val: etiquetas coinciden con B1
Entrenando XGBoost con 4000000 filas
✔ XGBoost: 1803 árboles | 2516.6s
✔ Modelo, configuración y predicciones de validación guardados


86

## 5. Predecir prueba, liberar datos y guardar probabilidades

In [10]:
test = load_compact("test")
y_test = test.pop(TARGET).to_numpy(dtype=np.uint8, copy=False)
check_b1_labels("test", y_test)
p_test = model.predict_proba(test, iteration_range=(0, best_iteration))[:, 1]
np.savez_compressed(OUT_DIR / "test_predictions.npz", y=y_test, p=p_test)
print("✔ Test:", len(y_test), "predicciones | probabilidad media:", round(float(p_test.mean()), 5))
del test, model, y_test, p_test
gc.collect()

  test: 20/20 fragmentos
✔ test: 1,000,000 filas | pandas 0.23 GiB | 2s
✔ test: etiquetas coinciden con B1
✔ Test: 1000000 predicciones | probabilidad media: 0.22942


36

## 6. Evaluar con las mismas métricas originales, sin reentrenar

Se recuperan únicamente los `npz` (etiquetas y probabilidades), que ocupan mucho menos que las matrices completas. El umbral que minimiza costo se decide exclusivamente con **validación**. Se maneja explícitamente la conversión de `numpy.int64` a JSON para no repetir el error ocurrido en B2.

In [11]:
sys.path.insert(0, str(Path("utils").resolve()))
import model_eval as me

with np.load(OUT_DIR / "val_predictions.npz") as z:
    y_val, p_val = z["y"], z["p"]
with np.load(OUT_DIR / "test_predictions.npz") as z:
    y_test, p_test = z["y"], z["p"]

threshold_val = float(me.best_cost_threshold(y_val, p_val, c_fp=COST_FP, c_fn=COST_FN))
metrics = {
    "val": me.evaluate_model("XGBoost", y_val, p_val, threshold_val, c_fp=COST_FP, c_fn=COST_FN),
    "test": me.evaluate_model("XGBoost", y_test, p_test, threshold_val, c_fp=COST_FP, c_fn=COST_FN),
    "threshold_from_validation": threshold_val,
    "threshold_theoretical": float(COST_FP/(COST_FP+COST_FN)),
}

def numpy_to_json(obj):
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.bool_): return bool(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    raise TypeError(f"No serializable: {type(obj).__name__}")

(OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False, default=numpy_to_json))
show_fields = ("roc_auc", "pr_auc", "precision", "recall", "f1", "brier", "log_loss", "costo_x1000")
print(json.dumps({s: {k: stats[k] for k in show_fields} for s, stats in metrics.items() if s in ("val", "test")},
                 indent=2, ensure_ascii=False, default=numpy_to_json))
required = ["model.ubj", "feature_schema.json", "run_config.json", "val_predictions.npz", "test_predictions.npz", "metrics.json"]
assert all((OUT_DIR / f).exists() for f in required), "Etapa B3 incompleta"
print("✔ B3 completada: modelo, esquema, configuración, predicciones y métricas")
del y_val, y_test, p_val, p_test
gc.collect()

{
  "val": {
    "roc_auc": 0.7546382137213997,
    "pr_auc": 0.5314668739918593,
    "precision": 0.4422983153745253,
    "recall": 0.5783775774121704,
    "f1": 0.5012667193331336,
    "brier": 0.1417641137486268,
    "log_loss": 0.44701449768123186,
    "costo_x1000": 444.106
  },
  "test": {
    "roc_auc": 0.7613812970032499,
    "pr_auc": 0.5472309597315087,
    "precision": 0.45535450417774914,
    "recall": 0.591406526015519,
    "f1": 0.5145388826169041,
    "brier": 0.1429225916931155,
    "log_loss": 0.4495928850642986,
    "costo_x1000": 443.209
  }
}
✔ B3 completada: modelo, esquema, configuración, predicciones y métricas


62

## 7. Respaldo del checkpoint B3 en S3

Solo subir cuando el paso 6 termine correctamente. El prefijo de B3 está separado de los checkpoints B1/B2 y **no publica todavía el modelo definitivo** para el endpoint de SageMaker.

In [1]:
from pathlib import Path
import boto3

# Configuración independiente
BUCKET = "bts-flights-analysis"
REGION = "us-east-1"

OUT_DIR = Path(
    "/home/ec2-user/SageMaker/outputs/"
    "outputs_modeling/checkpoints/xgboost_float64"
)

S3_PREFIX = (
    "modeling/checkpoints/"
    "xgboost-float64-original-seed42-h240"
)

REQUIRED = [
    "model.ubj",
    "feature_schema.json",
    "run_config.json",
    "val_predictions.npz",
    "test_predictions.npz",
    "metrics.json"
]

# 1. Verificar resultados guardados
print("Verificando archivos locales...\n")

missing = []

for name in REQUIRED:
    path = OUT_DIR / name

    if path.is_file() and path.stat().st_size > 0:
        size_mb = path.stat().st_size / 1024**2
        print(f"✔ {name}: {size_mb:.2f} MB")
    else:
        missing.append(name)
        print(f"✘ Falta: {name}")

if missing:
    raise RuntimeError(
        f"Checkpoint incompleto: {missing}. "
        "No es necesario repetir el entrenamiento "
        "si model.ubj ya existe."
    )

# 2. Crear nueva sesión AWS
session = boto3.Session(region_name=REGION)

if session.get_credentials() is None:
    raise RuntimeError(
        "No hay credenciales. Reinicia el Learner Lab."
    )

sts = session.client("sts")
print("\nIdentidad:", sts.get_caller_identity()["Arn"])

s3 = session.client("s3")

# 3. Respaldar todos los resultados
for name in REQUIRED:
    key = f"{S3_PREFIX}/{name}"

    s3.upload_file(
        str(OUT_DIR / name),
        BUCKET,
        key
    )

    print(f"Subido: s3://{BUCKET}/{key}")

print("\n✔ Checkpoint B3 float64 respaldado en S3")


Verificando archivos locales...

✔ model.ubj: 131.85 MB
✔ feature_schema.json: 0.13 MB
✔ run_config.json: 0.00 MB
✔ val_predictions.npz: 3.55 MB
✔ test_predictions.npz: 3.55 MB
✔ metrics.json: 0.00 MB

Identidad: arn:aws:sts::211125468543:assumed-role/LabRole/SageMaker
Subido: s3://bts-flights-analysis/modeling/checkpoints/xgboost-float64-original-seed42-h240/model.ubj
Subido: s3://bts-flights-analysis/modeling/checkpoints/xgboost-float64-original-seed42-h240/feature_schema.json
Subido: s3://bts-flights-analysis/modeling/checkpoints/xgboost-float64-original-seed42-h240/run_config.json
Subido: s3://bts-flights-analysis/modeling/checkpoints/xgboost-float64-original-seed42-h240/val_predictions.npz
Subido: s3://bts-flights-analysis/modeling/checkpoints/xgboost-float64-original-seed42-h240/test_predictions.npz
Subido: s3://bts-flights-analysis/modeling/checkpoints/xgboost-float64-original-seed42-h240/metrics.json

✔ Checkpoint B3 float64 respaldado en S3


## Siguiente etapa

Cuando veas **«✔ Checkpoint B3 de XGBoost respaldado en S3»**, podremos continuar con **B4 — K-means**, sobre las mismas muestras originales (1 M train, 1 M val, 1 M test), sin ejecutar de nuevo ingeniería de variables, LightGBM, regresión logística ni XGBoost. Posteriormente reuniremos las predicciones para reproducir las tablas comparativas, bootstrap pareado y figuras del modelado original.